## Process Lab Result Data

In [ ]:
import os

import pandas as pd
import numpy as np
import re

import random

from datetime import date

data_path = os.path.join("..", "data")
raw_data_path = os.path.join(data_path, 'raw_data/')
processed_data_path = os.path.join(data_path, 'processed_data/')

## Load Data

### Patients of Interest

In [ ]:
cols = ['master_person_id', 'inclusion_date', 'endpoint_date']

inclusion_patients_df = pd.read_csv(os.path.join(data_path, "opt_in_ckd_inclusion_patients.csv"))[cols]

del cols

inclusion_patients_df['inclusion_date'] = pd.to_datetime(inclusion_patients_df['inclusion_date']).dt.date
inclusion_patients_df['endpoint_date'] = pd.to_datetime(inclusion_patients_df['endpoint_date']).dt.date

### Social & Behavioural Data

In [ ]:
social_behavioural_df = pd.read_csv(os.path.join(raw_data_path, "20251205_social_behavioural_search_results.csv"))

social_behavioural_df['document_CreatedWhenDate'] = pd.to_datetime(social_behavioural_df['document_CreatedWhenDate'])

## Refined Social & Behavioural Data

In [ ]:
social_behavioural_refined_df = inclusion_patients_df.merge(social_behavioural_df, how='inner', on='master_person_id')

inclusion_date_filter = (social_behavioural_refined_df['document_CreatedWhenDate']>=social_behavioural_refined_df['inclusion_date'])
endpoint_date_filter = (social_behavioural_refined_df['document_CreatedWhenDate']<=social_behavioural_refined_df['endpoint_date'])

social_behavioural_refined_df = social_behavioural_refined_df[endpoint_date_filter].reset_index(drop=True)

del inclusion_date_filter, endpoint_date_filter

social_behavioural_refined_df.head()

In [ ]:
base_cols = ['master_person_id', 'document_CreatedWhenDate']

### Alcohol Classification Refinement

In [ ]:
def heavy_alcohol_use(classification):
    if not pd.isnull(classification):
        if classification in ['Occasional', 'Drinks Alcohol', 'Does Not Drink Alcohol']:
            return 'No'
        else:
            if int(classification) >= 15:
                return 'Yes'
            else:
                return 'No'
    else:
        return np.NaN

In [ ]:
alcohol_cols = base_cols[:]
alcohol_cols.append('alcohol_classification')

alcohol_classification_df = social_behavioural_refined_df[social_behavioural_refined_df['alcohol_classification'].notna()][alcohol_cols].reset_index(drop=True)

alcohol_classification_df['heavy_alcohol_use'] = alcohol_classification_df['alcohol_classification'].apply(lambda x: heavy_alcohol_use(x))

alcohol_classification_df = alcohol_classification_df[['master_person_id', 'heavy_alcohol_use']].drop_duplicates().reset_index(drop=True)

alcohol_classification_df = alcohol_classification_df.groupby('master_person_id').agg({'heavy_alcohol_use': list}).reset_index()
alcohol_classification_df['heavy_alcohol_use'] = alcohol_classification_df['heavy_alcohol_use'].apply(lambda x: 'Yes' if 'Yes' in x else 'No')

alcohol_classification_df.head()

### Smoking Status Refinement

In [ ]:
def smoking_status_detailing(smoking_status_list):
    last_smoking_status = smoking_status_list[-1]

    if last_smoking_status == 'Non-Smoker' and ('Current Smoker' in smoking_status_list[:-1] or 'Ex-Smoker' in smoking_status_list[:-1] or 'Occasional Smoker' in smoking_status_list[:-1]):
        return 'Ex-Smoker'
    else:
        return last_smoking_status

In [ ]:
smoking_cols = base_cols[:]
smoking_cols.append('smoking_status')

smoking_status_df = social_behavioural_refined_df[social_behavioural_refined_df['smoking_status'].notna()][smoking_cols].reset_index(drop=True)

smoking_status_refined_df = smoking_status_df.groupby('master_person_id').agg({'smoking_status': list}).reset_index()

del smoking_status_df

smoking_status_refined_df.columns = ['master_person_id', 'smoking_status_list']

smoking_status_refined_df['smoking_status'] = smoking_status_refined_df['smoking_status_list'].apply(lambda x: smoking_status_detailing(x))

del smoking_status_refined_df['smoking_status_list']

smoking_status_refined_df.head()

### Occupation Refinement

In [ ]:
def occupation(x):
    if x in ['Employed', 'Self Employed']:
        return 'Employed'
    elif x in ['Retired', 'Unemployed']:
        return x
    else:
        return 'Other'

In [ ]:
occupation_cols = base_cols[:]
occupation_cols.append('occupation')

occupation_df = social_behavioural_refined_df[social_behavioural_refined_df['occupation'].notna()][occupation_cols].reset_index(drop=True)

occupation_df = occupation_df.groupby('master_person_id').first().reset_index().drop(columns=['document_CreatedWhenDate'])
occupation_df['occupation'] = occupation_df['occupation'].apply(lambda x: occupation(x))

occupation_df.head()

### Living Status Refinement

In [ ]:
living_status_cols = base_cols[:]
living_status_cols.append('living_status')

living_status_df = social_behavioural_refined_df[social_behavioural_refined_df['living_status'].notna()][living_status_cols].reset_index(drop=True)

living_status_df = living_status_df.groupby('master_person_id').first().reset_index()

living_status_df['lives_alone'] = living_status_df['living_status'].apply(lambda x: 'Yes' if x=='Lives Alone' else 'No')

living_status_df = living_status_df.drop(columns=['document_CreatedWhenDate', 'living_status'])

living_status_df.head()

## Combine Social & Behavioural Datapoints

In [ ]:
join_col = 'master_person_id'

social_behavioural_df = inclusion_patients_df.merge(living_status_df, how='left', on=join_col).merge(occupation_df, how='left', on=join_col).merge(smoking_status_refined_df, how='left', on=join_col).merge(alcohol_classification_df, how='left', on=join_col)

social_behavioural_df.head()

## Export Social & Behavioural Data

In [ ]:
# --- Save Results ---
file_name = "20260306_social_behavioural_data.csv"

social_behavioural_df.to_csv(f"{processed_data_path}/{file_name}", index=False)
print("✅ Results saved.")